In [4]:
import numpy as np
import cv2

from model.net import DGCNet
def get_valid_pixel_mask(H, image_shape):
    h, w = image_shape[:2]

    Hinv = np.linalg.inv(H)

    y_grid, x_grid = np.indices((h, w))
    ones = np.ones_like(x_grid)
    coords = np.stack([x_grid, y_grid, ones], axis=-1).reshape(-1, 3).T  # Shape (3, N)

    mapped_coords = Hinv @ coords  # Shape (3, N)
    mapped_coords /= mapped_coords[2, :]  # Normalize by last row (homogeneous coords)

    x_mapped = mapped_coords[0, :]
    y_mapped = mapped_coords[1, :]

    valid = (
        (x_mapped >= 0) & (x_mapped < w) &
        (y_mapped >= 0) & (y_mapped < h)
    )

    # Convert to mask shape
    mask = valid.astype(np.uint8).reshape(h, w)
    return mask

import os
import os.path as osp
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import transforms
from tqdm import tqdm
import random

# --------- Constants and Model Setup ---------
IMG_SIZE_DGC = (240, 240)

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

dataset_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

class DeNormalize:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        for t, m, s in zip(tensor, self.mean, self.std):
            t.mul_(s).add_(m)
        return tensor

restore_image = DeNormalize(mean, std)

def make_identity_grid(size, device):
    h, w = size
    grid_y, grid_x = torch.meshgrid(torch.linspace(-1, 1, h, device=device),
                                    torch.linspace(-1, 1, w, device=device), indexing='ij')
    grid = torch.stack((grid_x, grid_y), 2)
    return grid.permute(2, 0, 1).unsqueeze(0)  # shape: (1, 2, H, W)

# --------- Load Model ---------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

net = DGCNet()
checkpoint = torch.load('/home/boat/proxyISP/DGC-Net/model/pretrained_models/dgc/checkpoint.pth', map_location=device)
net.load_state_dict(checkpoint['state_dict'])
net.eval().to(device)

def scale_homography(H, scaley, scalex):
    if H.shape != (3, 3):
        raise ValueError("Input must be a 3x3 matrix.")

    S = np.array([
        [scalex, 0,     0],
        [0,     scaley, 0],
        [0,     0,     1]
    ], dtype=np.float32)

    S_inv = np.linalg.inv(S)
    return S @ H @ S_inv

def homography_str_to_numpy(h_str):
    lines = h_str.strip().split('\n')
    matrix = [list(map(float, line.strip().split())) for line in lines]
    return np.array(matrix, dtype=np.float32)

def load_image(path):
    original = cv2.imread(path)
    if original is None:
        raise FileNotFoundError(f"Image not found: {path}")
    return original

def infer_and_warp(img1, img2):
    with torch.no_grad():
        tensor1 = dataset_transforms(img1).unsqueeze(0).to(device)
        tensor2 = dataset_transforms(img2).unsqueeze(0).to(device)
        flow_pyr, _ = net(tensor1, tensor2)
        grid = flow_pyr[-1].permute(0, 2, 3, 1)
        warped = F.grid_sample(tensor1, grid, align_corners=True)
        return restore_image(warped.squeeze()).clamp(0, 1).permute(1, 2, 0).cpu().numpy(), grid

def alpha_blend_images(img1, img2, alpha=0.5):
    return cv2.addWeighted(img1, alpha, img2, 1 - alpha, 0)

def plot_comparison(seq_name, target_index, src1, tgt1, warp1, gt1, gt2, src2, tgt2, warp2, valid_mask, save_dir=None):
    plt.figure(figsize=(20, 10))
    plt.suptitle(f'Sequence: {seq_name} | Pair: 1 → {target_index}', fontsize=16)

    titles = ['Source A', 'Target A', 'Warped A', 'Ground Truth 1', 'Overlay A vs GT1',
              'Source B', 'Target B', 'Warped B', 'Ground Truth 2', 'Overlay B vs GT2']

    warp1 = np.round((warp1 * 255)).astype(np.uint8)
    warp2 = np.round((warp2 * 255)).astype(np.uint8)

    if valid_mask is not None:
        valid_mask = valid_mask == 1
        warp1[~valid_mask] = 0
        warp2[~valid_mask] = 0
        gt1[~valid_mask] = 0
        gt2[~valid_mask] = 0

    # Convert GT and warped to pure channels for overlay
    overlay_a = np.zeros_like(gt1)
    overlay_a[..., 0] = warp1[..., 0]  # Blue channel from warped
    overlay_a[..., 2] = gt1[..., 0]    # Red channel from GT

    overlay_b = np.zeros_like(gt2)
    overlay_b[..., 0] = warp2[..., 0]  # Blue from warped
    overlay_b[..., 2] = gt2[..., 0]    # Red from GT

    images = [src1, tgt1, warp1, gt1, overlay_a,
              src2, tgt2, warp2, gt2, overlay_b]

    for i, img in enumerate(images):
        plt.subplot(2, 5, i + 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(titles[i])

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        save_path = osp.join(save_dir, f"{seq_name}_1to{target_index}_overlay.png")
        plt.savefig(save_path)
        print(f"Saved plot to {save_path}")

    plt.show()

def compute_homography_correctness_multi(
    hpatches_roots, num_sequences=5, shuffle=True,
    allowed_view_indices=None, save_dir=None, seq_names = None
):
    """
    hpatches_roots: list of dataset root paths
    """
    if len(hpatches_roots) < 2:
        print("Need at least 2 HPatches roots for comparison.")
        return

    # Find common sequence names across all roots
    seq_names_sets = [set(os.listdir(root)) for root in hpatches_roots]
    matched_seqs = sorted(list(set.intersection(*seq_names_sets)))

    if seq_names is not None:
        matched_seqs = seq_names

    if not matched_seqs:
        print("No matching sequences found.")
        return

    print(f"Found {len(matched_seqs)} matching sequences. Showing up to {num_sequences}.")

    if shuffle:
        random.shuffle(matched_seqs)

    if allowed_view_indices is None:
        allowed_view_indices = [2, 3, 4, 5, 6]

    result_dicts = {}
    for seq_name in tqdm(matched_seqs[:num_sequences]):
        imgs_original = []
        imgs_resized = []
        gt_imgs = []
        result_dicts[seq_name] = {}

        # Load the "1.ppm" source image for each dataset
        for root in hpatches_roots:
            path = osp.join(root, seq_name)
            try:
                img_original = load_image(osp.join(path, '1.ppm'))
                original_size = img_original.shape[:2]
                img = cv2.resize(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB), IMG_SIZE_DGC)
            except FileNotFoundError as e:
                print(f"Skipping sequence {seq_name}: {e}")
                continue
            imgs_original.append(img_original)
            imgs_resized.append(img)

        for i in range(2, 7):
            if i not in allowed_view_indices:
                continue

            tgt_imgs = []
            valid_masks = []
            warped_imgs = []
            gt_imgs = []

            result_dicts[seq_name][i] = {}

            for idx, root in enumerate(hpatches_roots):
                path = osp.join(root, seq_name)
                tgt_path = osp.join(path, f"{i}.ppm")
                if not osp.exists(tgt_path):
                    print(f"Missing {i}.ppm in {seq_name} for dataset {idx}, skipping.")
                    continue

                tgt_img_original = load_image(tgt_path)
                tgt_img = cv2.resize(cv2.cvtColor(tgt_img_original, cv2.COLOR_BGR2RGB), IMG_SIZE_DGC)
                tgt_imgs.append(tgt_img)

                # Load homography
                try:
                    with open(osp.join(path, f"H_1_{i}")) as f:
                        H = homography_str_to_numpy(f.read())
                except Exception as e:
                    print(f"Failed to read homography H_1_{i} in {seq_name} for dataset {idx}: {e}")
                    continue

                original_size = imgs_original[idx].shape[:2]
                orig_w = original_size[1]
                orig_h = original_size[0]
                eval_w = IMG_SIZE_DGC[0]
                eval_h = IMG_SIZE_DGC[1]
                sp_eval_size = [320, 240]
                sp_w = sp_eval_size[0]
                sp_h = sp_eval_size[1]
                
                corners4_src = np.array([
                    [0, 0],                                        # top-left
                    [0, sp_h - 1],                     # bottom-left
                    [sp_w - 1, sp_h - 1],  # bottom-right
                    [sp_w - 1, 0]                      # top-right
                ])
                
                valid_mask = get_valid_pixel_mask(H, tgt_img.shape)
                valid_masks.append(valid_mask)

                warped, grid = infer_and_warp(imgs_resized[idx], tgt_img)
                grid = grid.cpu().detach().numpy()[0]
                # print("grid max min", grid.max(), grid.min())
                grid = (grid + 1) * np.array([(eval_w - 1) / 2, (eval_h - 1) / 2])
                grid_flat = grid.reshape(-1, 2)
                x = np.arange(eval_w)
                y = np.arange(eval_h)
                X, Y = np.meshgrid(x, y)
                grid_flat_src = np.stack([X, Y], axis = -1).reshape(-1,2)
                
                estimated_H, inliers = cv2.findHomography(grid_flat,
                                                    grid_flat_src,
                                                    cv2.RANSAC)
                
                estimated_H = scale_homography(estimated_H, sp_h/eval_h, sp_w/eval_w)
                H = scale_homography(H, sp_h/orig_h, sp_w/orig_w)
                # H = np.linalg.inv(H)
                # print("grid_flat", grid_flat.shape, grid_flat)
                # print("grid_flat_src", grid_flat_src.shape, grid_flat_src)
                # print("estimated_H", estimated_H)
                # print("gt H", H)
                corners4_src_reshaped = corners4_src.reshape(-1, 1, 2).astype(np.float32)
                corners4_warp = cv2.perspectiveTransform(corners4_src_reshaped, estimated_H)  # warp using homography
                corners4_warp = corners4_warp.reshape(-1, 2)

                corners4_warp_gt = cv2.perspectiveTransform(corners4_src_reshaped, H)
                corners4_warp_gt = corners4_warp_gt.reshape(-1, 2)

                # print("corners4_src", corners4_src)
                # print("corners4_warp_gt", corners4_warp_gt)
                # print("corners4_warp", corners4_warp)

                dists = np.linalg.norm(corners4_warp - corners4_warp_gt, axis=1)
                avg_dist = dists.mean()
                correctness = avg_dist <= np.array([1, 3, 5, 10, 20, 50])
                # print(dists)
                # print(correctness)

                result_dicts[seq_name][i][root] = correctness
                
                # warped_imgs.append(warped)

                # print(imgs_original[idx].shape)
                # img_sp = cv2.resize(imgs_original[idx], (sp_w, sp_h))
                # warped = cv2.warpPerspective(img_sp, estimated_H, (sp_w, sp_h))
                # plt.imshow(warped[:, :, :])
                # plt.show()
    return result_dicts



# === PARAMETERS ===
HP_ROOTS = [i.path for i in os.scandir("/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC")]
hp_root3 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_ll_FIXZEROGRADBUG_CFANORMALIZE_train_v16.2-chroma-HumanTunedInitialHype_lowlight_pooled480x640_allHomoRepeatedRaw_standardize_lr0.0005_gradac32_123000_HpatchesV4"
hp_root2 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_ll_v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_lowlight_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_45000_HpatchesV4"
hp_root1 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_ll_v16.2-chroma-ISPDefaultInitialHype_original_HpatchesV4"
# hp_root3 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_sl_FIXZEROGRADBUG_CFANORMALIZE_train_v16.2-chroma-HumanTunedInitialHype_sunlit_pooled480x640_allHomoRepeatedRaw_standardize_lr0.0005_gradac32_105000_HpatchesV4"
# hp_root2 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_sl_v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_sunlit_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_120000_HpatchesV4"
# hp_root1 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_sl_v16.2-chroma-ISPDefaultInitialHype_original_HpatchesV4"

HP_ROOTS = [hp_root3,hp_root2,hp_root1]
# Call with filter (example: only viewpoints 2 and 5)
result_dicts = compute_homography_correctness_multi(
    HP_ROOTS,
    num_sequences=20,
    # allowed_view_indices=[2],
    # seq_names = ["ll_ais"],
    save_dir='/home/boat/proxyISP/DGC-Net/visualize_sl_HPatchesV4'
)



  0%|                                                                                                                                              | 0/16 [00:00<?, ?it/s]

Found 16 matching sequences. Showing up to 20.


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:45<00:00,  2.89s/it]


In [5]:
vizdict = {k: np.array([0,0,0,0,0,0]) for k in HP_ROOTS}
for hp_root in HP_ROOTS:
    num_pair = 0
    for seq in result_dicts.keys():
        for vp in range(2,7):
            correctness = result_dicts[seq][vp][hp_root].astype(np.uint8)
            vizdict[hp_root] += correctness
            num_pair += 1
            
vizdict = {k: v / num_pair for k,v in vizdict.items()}
correctnesses = np.array(list(vizdict.values()))
# correctnesses = correctnesses[:, :3]
print("mean", correctnesses.mean(axis = 1))
print("corr", correctnesses)

mean [0.2375     0.23333333 0.23541667]
corr [[0.     0.05   0.15   0.275  0.375  0.575 ]
 [0.     0.025  0.1125 0.275  0.3875 0.6   ]
 [0.     0.0375 0.125  0.275  0.3875 0.5875]]


====ll
mean [0.53958333 0.48958333 0.51666667]
corr [[0.0125 0.425  0.525  0.6875 0.75   0.8375]
 [0.0125 0.3375 0.4875 0.5875 0.7    0.8125]
 [0.025  0.4125 0.525  0.65   0.7    0.7875]]

====sl 
mean [0.53529412 0.52156863 0.5254902 ]
corr [[0.03529412 0.48235294 0.58823529 0.63529412 0.69411765 0.77647059]
 [0.05882353 0.45882353 0.52941176 0.63529412 0.67058824 0.77647059]
 [0.05882353 0.43529412 0.52941176 0.63529412 0.69411765 0.8       ]]
 

In [82]:
array([0.5372549 , 0.5254902 , 0.52352941])

NameError: name 'array' is not defined